In [1]:
import pandas as pd
from pathlib import Path

files = [f for f in Path("../data/raw/ipl").glob("*.csv")
         if not f.name.endswith("_info.csv")]
print("Number of matches found:", len(files))

balls = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
print(balls.shape)
balls.head()

Number of matches found: 1244


C:\Users\Envy\AppData\Local\Temp\ipykernel_30432\1020699325.py:8: DtypeWarning: Columns (1: season, 19: non_boundary, 26: fielder_3) have mixed types. Specify dtype option on import or set low_memory=False.
  balls = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)


(591464, 27)


,match_id,season,start_date,venue,innings,ball,actual_delivery,batting_team,bowling_team,striker,...,legbyes,penalty,non_boundary,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,fielder_1,fielder_2,fielder_3
0,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.1,0.1,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.2,0.2,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.3,0.3,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.4,0.4,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.5,0.5,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
balls.info()

<class 'pandas.DataFrame'>
RangeIndex: 591464 entries, 0 to 591463
Data columns (total 27 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   match_id                591464 non-null  int64  
 1   season                  591464 non-null  object 
 2   start_date              591464 non-null  str    
 3   venue                   591464 non-null  str    
 4   innings                 591464 non-null  int64  
 5   ball                    591464 non-null  float64
 6   actual_delivery         591464 non-null  float64
 7   batting_team            591464 non-null  str    
 8   bowling_team            591464 non-null  str    
 9   striker                 591464 non-null  str    
 10  non_striker             591464 non-null  str    
 11  bowler                  591464 non-null  str    
 12  runs_off_bat            591464 non-null  int64  
 13  extras                  591464 non-null  int64  
 14  wides                   19752 n

In [3]:
# Who has scored the most runs off the bat?
top_batters = (balls.groupby("striker")["runs_off_bat"]
                    .sum()
                    .sort_values(ascending=False)
                    .head(10))
print(top_batters)

# How many balls does the dataset contain per season?
print(balls.groupby("season").size())

# What are the most common ways of getting out?
print(balls["wicket_type"].value_counts())

striker
V Kohli           18692
RG Sharma         14662
S Dhawan          13538
DA Warner         13134
KL Rahul          11656
SK Raina          11072
MS Dhoni          10878
AM Rahane         10734
SV Samson         10362
AB de Villiers    10362
Name: runs_off_bat, dtype: int64
season
2009       13606
2011       17013
2012       23898
2013       36354
2014       28600
2015       27304
2016       28192
2017       27724
2018       28572
2019       18932
2021       20333
2022       35824
2023       35726
2024       34206
2025       24035
2026       17527
2007/08    26978
2009       13606
2009/10    28996
2011       17013
2012       11636
2019        9692
2020/21    29118
2021        8517
2025       10535
2026       17527
dtype: int64
wicket_type
caught                   18642
bowled                    4920
run out                   2388
lbw                       1766
caught and bowled          820
stumped                    776
hit wicket                  40
retired hurt                

In [4]:
print("unique match_ids:", balls["match_id"].nunique())
print("fully duplicated rows:", balls.duplicated().sum())
print(balls.groupby("match_id").size().describe())

unique match_ids: 1243
fully duplicated rows: 207206
count    1243.000000
mean      475.835881
std        46.417889
min        44.000000
25%       472.000000
50%       490.000000
75%       498.000000
max       538.000000
dtype: float64


In [5]:
# Does the same match appear under different file names?
print(sorted(f.name for f in files)[:6])
print(balls.groupby("match_id")["start_date"].first().value_counts().head(5))

['1082591.csv', '1082592.csv', '1082593.csv', '1082594.csv', '1082595.csv', '1082596.csv']
start_date
2008-04-19    2
2008-04-20    2
2008-04-26    2
2008-04-27    2
2008-05-01    2
Name: count, dtype: int64


In [6]:
balls["season"] = balls["season"].astype(str)
print(balls.groupby("season").size())

season
2007/08    26978
2009       27212
2009/10    28996
2011       34026
2012       35534
2013       36354
2014       28600
2015       27304
2016       28192
2017       27724
2018       28572
2019       28624
2020/21    29118
2021       28850
2022       35824
2023       35726
2024       34206
2025       34570
2026       35054
dtype: int64


In [7]:
# 1. Re-count duplicates now that season is fixed
print("duplicate rows:", balls.duplicated().sum(), "of", len(balls))

duplicate rows: 295732 of 591464


In [8]:
# 2. Look at ONE raw file: is it doubled at the source?
one = pd.read_csv(files[0])
print("rows in this file:", len(one))
print("duplicate rows inside it:", one.duplicated().sum())

rows in this file: 248
duplicate rows inside it: 0


In [9]:
# 3. You have 1244 files but 1243 match_ids. Which id appears in two files?
ids = pd.Series({f.name: pd.read_csv(f, usecols=["match_id"])["match_id"].iloc[0] for f in files})
print(ids[ids.duplicated(keep=False)])

1082591.csv        1082591
all_matches.csv    1082591
dtype: int64


In [10]:
balls = balls.drop_duplicates().reset_index(drop=True)
print(balls.shape)
print("balls per match:", len(balls) / balls["match_id"].nunique())


(295732, 27)
balls per match: 237.91794046661303


In [11]:
# Who has scored the most runs off the bat?
top_batters = (balls.groupby("striker")["runs_off_bat"]
                    .sum()
                    .sort_values(ascending=False)
                    .head(10))
print(top_batters)

striker
V Kohli           9346
RG Sharma         7331
S Dhawan          6769
DA Warner         6567
KL Rahul          5828
SK Raina          5536
MS Dhoni          5439
AM Rahane         5367
SV Samson         5181
AB de Villiers    5181
Name: runs_off_bat, dtype: int64
